# Build task 1 — Google Books API connection

**Purpose:** take a (title, author) pair from Katy's vision step, send a *targeted* query to the
Google Books volumes endpoint, and pull out the six fields the pipeline needs:
`title, authors, categories, averageRating, pageCount, publishedDate`.

Non-success cases (not found / request error / quota hit) print a clear message instead of crashing.
A half-second pause between requests keeps us under the free-tier rate limit.

The API key lives in `../api_key.txt`, which is listed in `.gitignore` and must **never** be pushed.

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

GOOGLE_BOOKS_URL = "https://www.googleapis.com/books/v1/volumes"

# The six volumeInfo fields our pipeline uses downstream
FIELDS = ["title", "authors", "categories", "averageRating", "pageCount", "publishedDate"]


def load_api_key(path="../api_key.txt"):
    """Read the Google Books API key from a local file kept out of git.

    Returns the key as a string, or None if the file is missing or still
    contains the placeholder. The volumes endpoint also works without a key,
    just with a lower anonymous quota, so None is not fatal.
    """
    key_file = Path(path)
    if not key_file.exists():
        print("No api_key.txt found - continuing without a key (lower quota).")
        return None
    key = key_file.read_text().strip()
    if not key or key.startswith("PASTE_"):
        print("api_key.txt still has the placeholder - continuing without a key.")
        return None
    return key


API_KEY = load_api_key()

## The connection

The query uses the `intitle:` and `inauthor:` operators so we ask for *that title by that author*,
not a loose keyword search. That is what gets us the right edition instead of a study guide or a
book that merely mentions the title.

### The response cache

Every confirmed answer is stored under a `(source, title, author)` key in
`data/api_cache.json`, so a book is only ever fetched from the network once. Confirmed
"not found" answers are cached too (as a marker), so a hallucinated title does not trigger
a fresh lookup on every run. Transient failures — errors, timeouts, quota replies — are
never cached. Besides saving time and quota, the cache makes runs **reproducible**: we
observed Google returning different editions of the same book on different days, and a
cached response pins the result.

In [21]:
# --- Response cache -----------------------------------------------------
# Division of labour agreed with Katy: the Google Books / Open Library
# metadata cache lives here in the API module; the genre-classification
# cache for her LLM outputs lives in her module.
CACHE_PATH = Path("../data/api_cache.json")
NOT_FOUND = "__NOT_FOUND__"  # marker for a confirmed catalogue miss


def load_cache(path=CACHE_PATH):
    """Load the response cache from disk, or start an empty one.

    One entry per (source, title, author): the raw record the source
    returned, or the NOT_FOUND marker when the source confirmed it has
    no such book. The JSON file survives kernel restarts, so a popular
    book is only ever fetched once - which saves quota and time, and
    makes reruns reproducible (live API results can differ between
    calls; cached ones cannot).
    """
    if path.exists():
        try:
            return json.loads(path.read_text())
        except ValueError:
            print("[CACHE] api_cache.json unreadable - starting fresh")
    return {"google": {}, "openlibrary": {}}


def save_cache(cache, path=CACHE_PATH):
    """Write the cache to disk (called after every new entry)."""
    path.write_text(json.dumps(cache))


def cache_key(title, author):
    """One key per book request: title and author, case-folded."""
    return f"{title.strip().lower()}|{author.strip().lower()}"


def cache_get(source, title, author):
    """Return the cached record, NOT_FOUND, or None if never fetched."""
    return API_CACHE[source].get(cache_key(title, author))


def cache_put(source, title, author, record):
    """Store a confirmed answer (a record or NOT_FOUND) and persist it.

    Errors, timeouts and quota replies are deliberately never cached:
    they are transient, and caching one would freeze a temporary
    failure into a permanent wrong answer.
    """
    API_CACHE[source][cache_key(title, author)] = record
    save_cache(API_CACHE)


API_CACHE = load_cache()
print(f"Cache loaded: {len(API_CACHE['google'])} Google Books entries, "
      f"{len(API_CACHE['openlibrary'])} Open Library entries")

Cache loaded: 2 Google Books entries, 0 Open Library entries


In [22]:
def pick_best_match(items, query_title):
    """Choose the most suitable edition from a list of API results.

    Google Books ranks results by the caller's locale, so from a Swiss IP
    the first hit is often a German translation (langRestrict does not
    reliably prevent this). Instead we fetch several results and score
    each one: +4 if the edition is in English, +2 if its title matches
    the query exactly, +1 if it has a page count. The highest-scoring
    item wins; ties go to Google's original ranking.
    """
    def score(item):
        vi = item.get("volumeInfo", {})
        s = 0
        if vi.get("language") == "en":
            s += 4
        if vi.get("title", "").strip().lower() == query_title.strip().lower():
            s += 2
        if vi.get("pageCount", 0) > 0:
            s += 1
        return s

    return max(items, key=score)


def fetch_google_books(title, author, api_key=None, pause=0.5, retries=2):
    """Look one book up on the Google Books volumes endpoint.

    Checks the response cache first: a cached record (or a cached
    confirmed miss) is returned immediately, with no pause and no
    network call. Only confirmed answers are written to the cache;
    errors and quota replies are not.

    Parameters
    ----------
    title : str
        Book title as guessed by the vision step.
    author : str
        Author name as guessed by the vision step.
    api_key : str or None
        Google Books API key. If None the request is sent anonymously.
    pause : float
        Seconds to sleep before each network request, to stay under
        the free-tier rate limit (~15 requests per minute).
    retries : int
        How many extra attempts to make on transient server errors
        (HTTP 5xx), waiting 2 seconds between attempts.

    Returns
    -------
    dict or None
        The volumeInfo record of the best-matching result (see
        pick_best_match), or None when the book was not found, the
        request failed, or the quota was hit. Each failure prints a
        clear message so we can see it in the run log.
    """
    cached = cache_get("google", title, author)
    if cached == NOT_FOUND:
        print(f"[CACHE] '{title}' is a known Google Books miss - no request sent")
        return None
    if cached is not None:
        return cached  # cache hit: no pause, no network call

    params = {
        "q": f'intitle:"{title}" inauthor:"{author}"',
        "maxResults": 10,  # fetch several editions, pick_best_match chooses
    }
    if api_key:
        params["key"] = api_key

    for attempt in range(retries + 1):
        time.sleep(pause)  # rate-limit courtesy pause

        try:
            response = requests.get(GOOGLE_BOOKS_URL, params=params, timeout=10)
        except requests.exceptions.RequestException as err:
            print(f"[ERROR] Request failed for '{title}' by {author}: {err}")
            return None

        if response.status_code >= 500:  # transient server error -> retry
            if attempt < retries:
                print(f"[RETRY] HTTP {response.status_code} for '{title}' - trying again...")
                time.sleep(2)
                continue
            print(f"[ERROR] HTTP {response.status_code} for '{title}' by {author} (gave up)")
            return None

        if response.status_code == 429 or (
            response.status_code == 403 and "quota" in response.text.lower()
        ):
            print(f"[QUOTA] Rate limit or daily quota reached at '{title}' - stop and retry later.")
            return None
        if response.status_code != 200:
            print(f"[ERROR] HTTP {response.status_code} for '{title}' by {author}")
            return None

        data = response.json()
        if data.get("totalItems", 0) == 0 or "items" not in data:
            print(f"[NOT FOUND] Google Books has no match for '{title}' by {author}")
            cache_put("google", title, author, NOT_FOUND)  # a confirmed miss is cacheable
            return None

        best = pick_best_match(data["items"], title)["volumeInfo"]
        cache_put("google", title, author, best)
        return best


def parse_volume_info(volume_info):
    """Reduce a raw volumeInfo record to the six fields the pipeline uses.

    Missing fields become None, so blanks are visible in the table
    (the missing-data policy in build task 3 deals with them).
    A pageCount of 0 also becomes None, because Google Books uses 0
    to mean 'unknown page count'. 'authors' and 'categories' arrive
    as lists and are joined with '; ' so each fits in one DataFrame cell.
    """
    row = {}
    for field in FIELDS:
        value = volume_info.get(field)
        if field == "pageCount" and value == 0:
            value = None  # Google Books uses 0 for 'unknown page count'
        if isinstance(value, list):
            value = "; ".join(str(v) for v in value)
        row[field] = value
    return row

## Smoke test: one well-known book, raw JSON first

Before parsing anything, look at the raw answer once so the shape of the JSON is familiar.

In [3]:
raw = fetch_google_books("The Hobbit", "J.R.R. Tolkien", api_key=API_KEY)
print(json.dumps(raw, indent=2)[:2000])  # first 2000 chars is enough to see the shape

{
  "title": "The Hobbit",
  "authors": [
    "J. R. R. Tolkien"
  ],
  "publisher": "HarperCollins UK",
  "publishedDate": "2009-04-20",
  "description": "This is the story of how a Baggins had an adventure, and found himself doing and saying things altogether unexpected... \u2018A flawless masterpiece\u2019 The Times",
  "industryIdentifiers": [
    {
      "type": "ISBN_13",
      "identifier": "9780007322602"
    },
    {
      "type": "ISBN_10",
      "identifier": "0007322607"
    }
  ],
  "readingModes": {
    "text": true,
    "image": false
  },
  "pageCount": 329,
  "printType": "BOOK",
  "categories": [
    "Fiction"
  ],
  "averageRating": 4.5,
  "ratingsCount": 101,
  "maturityRating": "NOT_MATURE",
  "allowAnonLogging": true,
  "contentVersion": "2.29.27.0.preview.2",
  "panelizationSummary": {
    "containsEpubBubbles": false,
    "containsImageBubbles": false
  },
  "imageLinks": {
    "smallThumbnail": "http://books.google.com/books/content?id=U799AY3yfqcC&printsec=fro

In [4]:
parse_volume_info(raw)

{'title': 'The Hobbit',
 'authors': 'J. R. R. Tolkien',
 'categories': 'Fiction',
 'averageRating': 4.5,
 'pageCount': 329,
 'publishedDate': '2009-04-20'}

## Run the twelve test books into a pandas table


In [5]:
TEST_BOOKS = [
    ("Red Rising", "Pierce Brown"),
    ("The Secret History", "Donna Tartt"),
    ("Klara and the Sun", "Kazuo Ishiguro"),
    ("Piranesi", "Susanna Clarke"),
    ("Normal People", "Sally Rooney"),
    ("Rebecca", "Daphne du Maurier"),
    ("Sapiens", "Yuval Noah Harari"),
    ("The Alchemist", "Paulo Coelho"),
    ("Flowers for Algernon", "Daniel Keyes"),
    ("The Penelopiad", "Margaret Atwood"),
    ("Beloved", "Toni Morrison"),
    ("A Fine Balance", "Rohinton Mistry"), 
]
 


def build_metadata_table(books, api_key=None):
    """Fetch metadata for a list of (title, author) pairs.

    Returns a DataFrame with one row per book: the original query columns
    plus the six Google Books fields, plus a 'found' flag. Books that were
    not found keep their row (all fields None) so the coverage count in
    phase 3 stays honest.
    """
    rows = []
    for title, author in books:
        volume_info = fetch_google_books(title, author, api_key=api_key)
        row = {"query_title": title, "query_author": author, "found": volume_info is not None}
        row.update(parse_volume_info(volume_info) if volume_info else {f: None for f in FIELDS})
        rows.append(row)
    return pd.DataFrame(rows)


books_df = build_metadata_table(TEST_BOOKS, api_key=API_KEY)
books_df

,query_title,query_author,found,title,authors,categories,averageRating,pageCount,publishedDate
0,Red Rising,Pierce Brown,True,Red Rising,Pierce Brown,Fiction,NaN,NaN,2014-07-15
1,The Secret History,Donna Tartt,True,The Secret History,Donna Tartt,Classicists,NaN,628.0,2007
2,Klara and the Sun,Kazuo Ishiguro,True,Klara and the Sun,Kazuo Ishiguro,Fiction,NaN,319.0,2021-03-02
3,Piranesi,Susanna Clarke,True,Piranesi,Susanna Clarke,Fiction,NaN,231.0,2020-09-15
4,Normal People,Sally Rooney,True,Normal People,Sally Rooney,College students,NaN,300.0,2019
5,Rebecca,Daphne du Maurier,True,Rebecca,Daphne Du Maurier,Drama,4.5,420.0,2006-09-05
6,Sapiens,Yuval Noah Harari,True,Sapiens,Yuval Noah Harari,History,4.5,354.0,2014-09-04
7,The Alchemist,Paulo Coelho,True,The Alchemist,Paulo Coelho,None,NaN,177.0,2002
8,Flowers for Algernon,Daniel Keyes,True,Flowers for Algernon,Daniel Keyes,Brain,NaN,274.0,1966
9,The Penelopiad,Margaret Atwood,True,The Penelopiad,Margaret Atwood,Performing Arts,NaN,118.0,2014-10-23


## First look at coverage (feeds phase 3)

How many of the twelve books came back with each field filled — these percentages go
straight into the notes and later into Results and Discussion.

In [6]:
coverage = books_df[FIELDS].notna().mean().mul(100).round(1)
print("Field coverage across the test books (%):")
print(coverage.to_string())

books_df.to_csv("../data/google_books_test_run.csv", index=False)
print("\nSaved to ../data/google_books_test_run.csv")

Field coverage across the test books (%):
title            100.0
authors          100.0
categories        91.7
averageRating     33.3
pageCount         91.7
publishedDate    100.0

Saved to ../data/google_books_test_run.csv


# Build task 2 — Open Library fallback

No single source is complete: in our Google-only run, `categories` came back blank for one
book in twelve and `averageRating` for more than half. Open Library — a free, public
catalogue run by the Internet Archive — is the second source.

**Fallback rule (exact trigger):** Open Library is called only when

1. Google Books found no book at all, **or**
2. Google Books found the book but left `pageCount`, `categories` or `publishedDate` blank.

**Tie-break rule (order of trust):** Google Books wins. The fallback only fills fields that are
still blank and never overwrites a value Google already gave us. (Written down here so the
methodology section can quote it verbatim.)

Open Library's search endpoint (`https://openlibrary.org/search.json`) takes separate `title`
and `author` parameters. Its first hit is not always the right edition — for *The Alchemist* it
returns a graphic novel first — so we score several results and pick the best, the same idea
as `pick_best_match` for Google Books. Open Library has no average rating, which is fine:
that field is display-only anyway.

In [ ]:
OPEN_LIBRARY_URL = "https://openlibrary.org/search.json"

# The Open Library fields we request, and (below) how they map onto our names
OL_FIELDS = "title,author_name,subject,number_of_pages_median,first_publish_year"


def pick_best_ol_doc(docs, query_title, query_author):
    """Choose the most suitable record from Open Library search results.

    Open Library's first hit is not always the right book (searching
    'The Alchemist' returns a graphic novel first). Score each result:
    +4 if the author's surname appears in the record's author list,
    +2 if the title matches the query exactly, +1 if a page count is
    present. Highest score wins; ties keep Open Library's own ranking.
    """
    def score(doc):
        s = 0
        authors = " ".join(doc.get("author_name", [])).lower()
        if query_author.split()[-1].lower() in authors:  # surname is the stable part
            s += 4
        if doc.get("title", "").strip().lower() == query_title.strip().lower():
            s += 2
        if doc.get("number_of_pages_median"):
            s += 1
        return s

    return max(docs, key=score)


def fetch_open_library(title, author, pause=1.0, retries=2):
    """Look one book up on the Open Library search endpoint.

    Same contract as fetch_google_books, including the cache check
    first: a cached record or confirmed miss returns immediately with
    no pause and no network call, and only confirmed answers are ever
    cached. Open Library is a donation-funded service and asks
    automated users to be gentle, hence the full one-second pause
    before real requests. It is also slower and flakier than Google:
    read timeouts and, when throttling, HTML replies instead of JSON
    both happen in practice, so both are treated as transient errors
    and retried.
    """
    cached = cache_get("openlibrary", title, author)
    if cached == NOT_FOUND:
        print(f"[CACHE] '{title}' is a known Open Library miss - no request sent")
        return None
    if cached is not None:
        return cached  # cache hit: no pause, no network call

    params = {"title": title, "author": author, "fields": OL_FIELDS, "limit": 10}

    for attempt in range(retries + 1):
        time.sleep(pause)  # be polite to a free service

        try:
            response = requests.get(OPEN_LIBRARY_URL, params=params, timeout=20)
        except requests.exceptions.RequestException as err:
            if attempt < retries:  # timeouts are usually transient here
                print(f"[OL RETRY] Request failed for '{title}' - trying again...")
                time.sleep(2)
                continue
            print(f"[OL ERROR] Request failed for '{title}' by {author}: {err}")
            return None

        if response.status_code >= 500 or response.status_code == 429:
            if attempt < retries:
                print(f"[OL RETRY] HTTP {response.status_code} for '{title}' - trying again...")
                time.sleep(2)
                continue
            print(f"[OL ERROR] HTTP {response.status_code} for '{title}' by {author} (gave up)")
            return None
        if response.status_code != 200:
            print(f"[OL ERROR] HTTP {response.status_code} for '{title}' by {author}")
            return None

        try:
            docs = response.json().get("docs", [])
        except ValueError:  # throttling can return HTML instead of JSON
            if attempt < retries:
                print(f"[OL RETRY] Non-JSON reply for '{title}' - trying again...")
                time.sleep(2)
                continue
            print(f"[OL ERROR] Non-JSON reply for '{title}' by {author} (gave up)")
            return None

        if not docs:
            print(f"[OL NOT FOUND] Open Library has no match for '{title}' by {author}")
            cache_put("openlibrary", title, author, NOT_FOUND)  # confirmed miss
            return None

        best = pick_best_ol_doc(docs, title, author)
        cache_put("openlibrary", title, author, best)
        return best


def parse_ol_doc(doc):
    """Translate an Open Library record into our Google Books field names.

    pageCount comes from number_of_pages_median (the median page count
    across all editions Open Library knows - a sensible single number),
    publishedDate from first_publish_year (the original publication year,
    not the year of some later reprint). Subject tags can run to dozens,
    so we keep the first three. There is no averageRating key: Open
    Library has no ratings, and the field is display-only anyway.
    """
    subjects = doc.get("subject") or []
    return {
        "title": doc.get("title"),
        "authors": "; ".join(doc.get("author_name", [])) or None,
        "categories": "; ".join(subjects[:3]) or None,
        "pageCount": doc.get("number_of_pages_median"),
        "publishedDate": doc.get("first_publish_year"),
    }

In [8]:
# Smoke test: one book Open Library must rescue from a bad first hit
# (the graphic novel), and one where it knows the original edition.
# A lookup can legitimately fail (timeout, throttling), so guard the None.
for smoke_title, smoke_author in [("The Alchemist", "Paulo Coelho"), ("Red Rising", "Pierce Brown")]:
    doc = fetch_open_library(smoke_title, smoke_author)
    print(parse_ol_doc(doc) if doc else f"-> no usable answer for '{smoke_title}', see message above")

{'title': 'The Alchemist', 'authors': 'Paulo Coelho (Author)', 'categories': None, 'pageCount': 386, 'publishedDate': 2010}
{'title': 'Red Rising', 'authors': 'Pierce Brown', 'categories': 'franchise:Red Rising; series:Red Rising Trilogy; series:Red Rising Saga', 'pageCount': 442, 'publishedDate': 2014}


In [9]:
# Fields the fallback may fill. averageRating stays out: it is display-only
# and Open Library does not provide one anyway.
FALLBACK_FIELDS = ["pageCount", "categories", "publishedDate"]


def apply_fallback(df):
    """Fill the holes Google Books left, using Open Library.

    Trigger rule: a book qualifies when Google found nothing at all, or
    when pageCount, categories or publishedDate is blank. Books whose
    Google record is complete are skipped, so we never double the
    request count for no reason.

    Tie-break rule: Google Books wins. Only blank fields are filled;
    existing values are never overwritten.

    Returns a copy of the DataFrame with the blanks filled where
    possible, plus a 'source' column ('google', 'google+openlibrary'
    or 'openlibrary') so every value's origin stays traceable. Prints
    one line per fallback call so the run log shows what happened.
    """
    df = df.copy()
    df["source"] = df["found"].map({True: "google", False: None})

    for idx, row in df.iterrows():
        blanks = [f for f in FALLBACK_FIELDS if pd.isna(row[f])]
        if row["found"] and not blanks:
            continue  # Google delivered everything - no request wasted

        doc = fetch_open_library(row["query_title"], row["query_author"])
        if doc is None:
            continue  # both sources failed; build task 3 decides what happens

        ol_record = parse_ol_doc(doc)
        # A book Google missed entirely takes every field Open Library has;
        # otherwise only the blanks may be filled (tie-break rule).
        targets = FIELDS if not row["found"] else blanks
        filled = []
        for field in targets:
            if ol_record.get(field) is not None and pd.isna(row[field]):
                df.at[idx, field] = ol_record[field]
                filled.append(field)

        if filled:
            df.at[idx, "source"] = "google+openlibrary" if row["found"] else "openlibrary"
            df.at[idx, "found"] = True
            print(f"[FALLBACK] '{row['query_title']}': filled {', '.join(filled)} from Open Library")
        else:
            print(f"[FALLBACK] '{row['query_title']}': Open Library had nothing new to add")

    return df


books_full_df = apply_fallback(books_df)
books_full_df

[FALLBACK] 'Red Rising': filled pageCount from Open Library
[FALLBACK] 'The Alchemist': Open Library had nothing new to add


,query_title,query_author,found,title,authors,categories,averageRating,pageCount,publishedDate,source
0,Red Rising,Pierce Brown,True,Red Rising,Pierce Brown,Fiction,NaN,442.0,2014-07-15,google+openlibrary
1,The Secret History,Donna Tartt,True,The Secret History,Donna Tartt,Classicists,NaN,628.0,2007,google
2,Klara and the Sun,Kazuo Ishiguro,True,Klara and the Sun,Kazuo Ishiguro,Fiction,NaN,319.0,2021-03-02,google
3,Piranesi,Susanna Clarke,True,Piranesi,Susanna Clarke,Fiction,NaN,231.0,2020-09-15,google
4,Normal People,Sally Rooney,True,Normal People,Sally Rooney,College students,NaN,300.0,2019,google
5,Rebecca,Daphne du Maurier,True,Rebecca,Daphne Du Maurier,Drama,4.5,420.0,2006-09-05,google
6,Sapiens,Yuval Noah Harari,True,Sapiens,Yuval Noah Harari,History,4.5,354.0,2014-09-04,google
7,The Alchemist,Paulo Coelho,True,The Alchemist,Paulo Coelho,None,NaN,177.0,2002,google
8,Flowers for Algernon,Daniel Keyes,True,Flowers for Algernon,Daniel Keyes,Brain,NaN,274.0,1966,google
9,The Penelopiad,Margaret Atwood,True,The Penelopiad,Margaret Atwood,Performing Arts,NaN,118.0,2014-10-23,google


## Coverage before vs. after the fallback

The same six-field count as in phase 3, now side by side — the "gain" column is the
improvement the fallback bought us. These numbers (plus the run log above showing which
books triggered the fallback) go to Results and Discussion.

In [11]:
coverage_before = books_df[FIELDS].notna().mean().mul(100).round(1)
coverage_after = books_full_df[FIELDS].notna().mean().mul(100).round(1)

comparison = pd.DataFrame({
    "google only (%)": coverage_before,
    "with fallback (%)": coverage_after,
})
comparison["gain (pp)"] = (comparison["with fallback (%)"] - comparison["google only (%)"]).round(1)
print(comparison.to_string())
print("\nNote: twelve books is a small sample - each book moves a field by 8.3")
print("percentage points, so treat these numbers as indicative, not precise.")

books_full_df.to_csv("../data/metadata_with_fallback.csv", index=False)
print("\nSaved to ../data/metadata_with_fallback.csv")

               google only (%)  with fallback (%)  gain (pp)
title                    100.0              100.0        0.0
authors                  100.0              100.0        0.0
categories                91.7               91.7        0.0
averageRating             33.3               33.3        0.0
pageCount                 91.7              100.0        8.3
publishedDate            100.0              100.0        0.0

Note: twelve books is a small sample - each book moves a field by 8.3
percentage points, so treat these numbers as indicative, not precise.

Saved to ../data/metadata_with_fallback.csv


**Test if it works**

In [13]:
print("Hello")

Hello


# Build task 3 — Missing metadata handling

Even after the fallback some fields stay blank (our *The Alchemist* category, for example),
and in real shelf photos some books will not be found at all. A nearest-neighbour distance
cannot be computed against a blank, so every gap gets a **written rule**:

1. **Drop unmatched books.** A book neither source could confirm is removed — no
   half-empty card, no guessed entry. The number of drops is counted and reported: it
   measures how often the vision step "hallucinated" a title, one of the group's
   evaluation metrics.
2. **Impute missing numerics with the median.** A blank `pageCount` (or year) is filled
   with the median of the books we do have. *Imputation* means filling a blank with a
   declared, sensible estimate. The median (middle value) is used instead of the mean
   because one 930-page doorstop like *A Fine Balance* would drag a mean upward and
   distort every fill. Each fill is flagged in a `*_imputed` column so it stays visible.
3. **Never impute `averageRating`.** It is present for only a third of our books; filling
   the other two thirds would invent more data than we measured. It is shown in the app
   where it exists and stays out of the feature vector entirely.

A blank `categories` needs no rule here: the genre used in the feature vector comes from
Katy's classifier, not from the raw API category.

One practical step first: `publishedDate` arrives in mixed formats (`"2014-07-15"`,
`"2007"`, or a plain Open Library integer year), so the four-digit **year** is extracted
into its own numeric column — that number is what build task 4 standardises.

In [14]:
import re


def extract_year(published_date):
    """Pull a four-digit year out of whatever the sources sent.

    Google Books delivers strings like '2014-07-15' or '2007', Open
    Library a plain integer year. The feature vector needs one number,
    so everything is reduced to an int - or None when no year is there.
    """
    if pd.isna(published_date):
        return None
    match = re.search(r"\d{4}", str(published_date))
    return int(match.group()) if match else None


def apply_missing_data_policy(df):
    """Apply the three written missing-data rules, in order.

    Rule 1 - drop unmatched books. A book neither source could confirm
        is removed and counted; the drop count measures how often the
        vision step hallucinated a title (a group evaluation metric).
    Rule 2 - impute missing numerics with the median. A blank pageCount
        or year is filled with the median of the books we do have (the
        median resists outliers where the mean does not). Every fill is
        flagged in a *_imputed column so nothing happens silently.
    Rule 3 - never impute averageRating. Present for only a third of
        books, so it passes through untouched: displayed where it
        exists, excluded from the feature vector.

    Returns
    -------
    (pandas.DataFrame, int)
        The cleaned table, and the number of dropped books for the
        Results and Discussion section.
    """
    df = df.copy()

    # Rule 1: drop and count the books no source could confirm
    dropped = int((~df["found"]).sum())
    df = df[df["found"]].copy()

    # publishedDate arrives in mixed formats; the vector needs a number
    df["year"] = df["publishedDate"].map(extract_year)

    # Rule 2: median imputation, flagged so every fill stays visible
    for col in ["pageCount", "year"]:
        median_value = df[col].median()
        df[f"{col}_imputed"] = df[col].isna()
        df[col] = df[col].fillna(median_value)
        print(f"{col}: median = {median_value:.0f}, "
              f"imputed {int(df[f'{col}_imputed'].sum())} value(s)")

    # Rule 3: averageRating passes through untouched
    print(f"Dropped books (no source could confirm them): {dropped}")

    return df, dropped


books_clean_df, dropped_count = apply_missing_data_policy(books_full_df)
books_clean_df[["query_title", "pageCount", "pageCount_imputed",
                "year", "year_imputed", "averageRating", "source"]]

pageCount: median = 312, imputed 0 value(s)
year: median = 2012, imputed 0 value(s)
Dropped books (no source could confirm them): 0


,query_title,pageCount,pageCount_imputed,year,year_imputed,averageRating,source
0,Red Rising,442.0,False,2014,False,NaN,google+openlibrary
1,The Secret History,628.0,False,2007,False,NaN,google
2,Klara and the Sun,319.0,False,2021,False,NaN,google
3,Piranesi,231.0,False,2020,False,NaN,google
4,Normal People,300.0,False,2019,False,NaN,google
5,Rebecca,420.0,False,2006,False,4.5,google
6,Sapiens,354.0,False,2014,False,4.5,google
7,The Alchemist,177.0,False,2002,False,NaN,google
8,Flowers for Algernon,274.0,False,1966,False,NaN,google
9,The Penelopiad,118.0,False,2014,False,NaN,google


In [15]:
books_clean_df.to_csv("../data/metadata_clean.csv", index=False)
print("Saved to ../data/metadata_clean.csv")

print(f"\nFor Results and Discussion: {dropped_count} of {len(books_full_df)} books were")
print("dropped because neither source could confirm them. On this test set every")
print("title is a real book, so 0 drops is the expected (and correct) outcome -")
print("the counter proves its worth once real vision-step output flows in.")

Saved to ../data/metadata_clean.csv

For Results and Discussion: 0 of 12 books were
dropped because neither source could confirm them. On this test set every
title is a real book, so 0 drops is the expected (and correct) outcome -
the counter proves its worth once real vision-step output flows in.


# Extension — the thirty-book test

Twelve books is a small sample: one book moves every percentage by 8.3 points. This run
re-measures coverage on **30 real books plus 2 invented ones** so the reported numbers are
less noisy (each real book now moves a field by ~3.3 points).

The list is deliberately mixed, because each slice stresses a different part of the pipeline:

- **mainstream bestsellers** — the easy case, both sources should score;
- **translated fiction** — stresses the English-edition scoring in `pick_best_match`;
- **older / less commercial titles** — where Open Library's catalogue should show its
  strength over Google's;
- **two invented books** — no source can confirm them, so the drop counter and the
  `[NOT FOUND]` path finally run for real. Their drop rate stands in for the vision step's
  hallucination metric.

Field coverage is measured on *found* books only, so the two invented titles do not drag
every field down by the same constant amount — they are reported separately as drops.

⏱️ Expect the run to take **2–3 minutes**: 32 Google calls at half a second pause each,
plus a fallback call (a full second, politeness pause) for every book with a gap.

In [16]:
# 30 real books + 2 invented ones. The first twelve are the group's
# original test set, so results stay comparable with the earlier runs.
EXTENDED_TEST_BOOKS = TEST_BOOKS + [
    # mainstream bestsellers - the easy case
    ("The Hobbit", "J.R.R. Tolkien"),
    ("Dune", "Frank Herbert"),
    ("The Martian", "Andy Weir"),
    ("Gone Girl", "Gillian Flynn"),
    ("The Name of the Wind", "Patrick Rothfuss"),
    ("Educated", "Tara Westover"),
    ("Thinking, Fast and Slow", "Daniel Kahneman"),
    ("A Brief History of Time", "Stephen Hawking"),
    # translated fiction - stresses the English-edition scoring
    ("Kafka on the Shore", "Haruki Murakami"),
    ("My Brilliant Friend", "Elena Ferrante"),
    ("The Shadow of the Wind", "Carlos Ruiz Zafon"),
    ("Perfume", "Patrick Suskind"),
    ("The Master and Margarita", "Mikhail Bulgakov"),
    # older / less commercial - where Open Library should shine
    ("Things Fall Apart", "Chinua Achebe"),
    ("Wide Sargasso Sea", "Jean Rhys"),
    ("Stoner", "John Williams"),
    ("Excellent Women", "Barbara Pym"),
    ("The Towers of Trebizond", "Rose Macaulay"),
    # invented books - no catalogue can confirm these, so they must
    # end up in the drop count, not in the results
    ("Definitely Not A Real Book", "Nobody Nowhere"),
    ("The Invisible Shelf Mystery", "A. N. Hallucination"),
]

print(f"Running the full pipeline on {len(EXTENDED_TEST_BOOKS)} books "
      f"({len(EXTENDED_TEST_BOOKS) - 2} real + 2 invented)...\n")

# The whole chain: Google Books -> Open Library fallback -> missing-data policy
extended_df = build_metadata_table(EXTENDED_TEST_BOOKS, api_key=API_KEY)
print()
extended_full_df = apply_fallback(extended_df)
print()
extended_clean_df, extended_dropped = apply_missing_data_policy(extended_full_df)

extended_clean_df[["query_title", "categories", "pageCount", "pageCount_imputed",
                   "year", "year_imputed", "source"]]

Running the full pipeline on 32 books (30 real + 2 invented)...

[NOT FOUND] Google Books has no match for 'Definitely Not A Real Book' by Nobody Nowhere
[NOT FOUND] Google Books has no match for 'The Invisible Shelf Mystery' by A. N. Hallucination

[FALLBACK] 'Red Rising': filled pageCount from Open Library
[FALLBACK] 'The Alchemist': Open Library had nothing new to add
[FALLBACK] 'Educated': filled categories from Open Library
[FALLBACK] 'Perfume': filled categories from Open Library
[OL NOT FOUND] Open Library has no match for 'Definitely Not A Real Book' by Nobody Nowhere
[OL NOT FOUND] Open Library has no match for 'The Invisible Shelf Mystery' by A. N. Hallucination

pageCount: median = 326, imputed 0 value(s)
year: median = 2010, imputed 0 value(s)
Dropped books (no source could confirm them): 2


,query_title,categories,pageCount,pageCount_imputed,year,year_imputed,source
0,Red Rising,Fiction,442.0,False,2014,False,google+openlibrary
1,The Secret History,Classicists,628.0,False,2007,False,google
2,Klara and the Sun,Fiction,319.0,False,2021,False,google
3,Piranesi,Fiction,231.0,False,2020,False,google
4,Normal People,Fiction,271.0,False,2018,False,google
5,Rebecca,Drama,420.0,False,2006,False,google
6,Sapiens,History,354.0,False,2014,False,google
7,The Alchemist,None,177.0,False,2002,False,google
8,Flowers for Algernon,Fiction,324.0,False,2004,False,google
9,The Penelopiad,Performing Arts,118.0,False,2014,False,google


In [17]:
n_invented = 2
n_real = len(EXTENDED_TEST_BOOKS) - n_invented

print(f"Found by Google alone:       {int(extended_df['found'].sum())} of {len(extended_df)}")
print(f"Found after the fallback:    {int(extended_full_df['found'].sum())} of {len(extended_full_df)}")
print(f"Dropped (no source at all):  {extended_dropped} "
      f"(expected: the {n_invented} invented books)\n")

# Field coverage among found books, before vs. after the fallback
cov_google = extended_df.loc[extended_df["found"], FIELDS].notna().mean().mul(100).round(1)
cov_fallback = extended_full_df.loc[extended_full_df["found"], FIELDS].notna().mean().mul(100).round(1)

ext_comparison = pd.DataFrame({
    "google only (%)": cov_google,
    "with fallback (%)": cov_fallback,
})
ext_comparison["gain (pp)"] = (ext_comparison["with fallback (%)"]
                               - ext_comparison["google only (%)"]).round(1)
print(ext_comparison.to_string())

# Which books needed the second source? Worth a sentence in the report.
rescued = extended_full_df[extended_full_df["found"]
                           & (extended_full_df["source"] != "google")]
print(f"\nBooks that needed Open Library ({len(rescued)}):")
print(rescued[["query_title", "source"]].to_string(index=False))

extended_clean_df.to_csv("../data/metadata_clean_30books.csv", index=False)
print("\nSaved to ../data/metadata_clean_30books.csv")

Found by Google alone:       30 of 32
Found after the fallback:    30 of 32
Dropped (no source at all):  2 (expected: the 2 invented books)

               google only (%)  with fallback (%)  gain (pp)
title                    100.0              100.0        0.0
authors                  100.0              100.0        0.0
categories                90.0               96.7        6.7
averageRating             43.3               43.3        0.0
pageCount                 96.7              100.0        3.3
publishedDate            100.0              100.0        0.0

Books that needed Open Library (3):
query_title             source
 Red Rising google+openlibrary
   Educated google+openlibrary
    Perfume google+openlibrary

Saved to ../data/metadata_clean_30books.csv


# Build task 4 — Feature vector construction

A **feature vector** is a row of numbers describing one thing in a form a computer can
compare. Each book becomes one vector; the user's taste
profile is a small pile of these vectors, one per liked book.

**The four features, in the exact order the ranker expects:**

| # | Feature | Type | Encoding |
|---|---------|------|----------|
| 1 | `genre_index` | category | index 0–9 into the genre similarity matrix |
| 2 | `known_author` | binary | 1.0 if the author is already in the saved profile, else 0.0 |
| 3 | `pageCount_z` | numeric | standardised page count |
| 4 | `year_z` | numeric | standardised publication year |

`averageRating` is deliberately left out: present for only ~a third of books (33–43 % in
our runs), so imputing it would invent more data than we measured. Shown on screen,
excluded from the maths.

**Genre — similarity matrix instead of one-hot.** The ten genre labels (exact spelling
agreed with Katy) are listed in the code below; their list position is their index 0–9.
Genre similarity comes from the 10×10 matrix: 1.0 on the diagonal (same genre),
0.1 default for unrelated pairs, hand-set values for related pairs (e.g. Biography ↔
History 0.7). This fixes the known weakness of one-hot encoding, which would treat
every genre as equally far from every other — as if Science Fiction were as distant
from Fantasy as from Romance. One-hot + Euclidean remains the considered alternative;
the matrix was chosen because it encodes how reading taste actually crosses genres.

**Standardisation for the numbers.** Formula: **z = (x − μ) / σ**, where *x* is the raw
value (e.g. 384 pages), *μ* is the mean of that column, and *σ* is its standard
deviation (how spread out the column is). The z-score says "how many standard
deviations above or below average is this book", which puts pages (hundreds) and years
(thousands) on the same footing — otherwise the year would dominate every distance
simply because its numbers are bigger. The scaler is **fitted once on the profile**
and reused for every shelf book, so all vectors share one scale between uploads.

In [18]:
import numpy as np

# The ten genre labels, exact spelling confirmed with Katy. The list
# position of each label is its index 0-9, and that index order must
# match Alexa's similarity matrix below - do not reorder either side.
GENRE_LABELS = [
    "Romance",                          # 0
    "Mystery, Thriller & Crime",        # 1
    "Science Fiction & Fantasy",        # 2
    "Children",                         # 3
    "Young Adult",                      # 4
    "Biography & Memoir",               # 5
    "History & Politics",               # 6
    "Science, Tech & Nature",           # 7
    "Self-Help & Lifestyle",            # 8
    "General & Contemporary Fiction",   # 9
]

GENRE_TO_INDEX = {label: i for i, label in enumerate(GENRE_LABELS)}


def _build_similarity_matrix():
    """Alexa's genre similarity matrix (agreed group code, do not edit).

    10x10, symmetric: 1.0 on the diagonal (same genre), 0.1 default for
    unrelated pairs, hand-set values for genres that share readers.
    Her ranker uses it to score the genre part of the distance between
    two books via their genre indices.
    """
    # Start with 0.1 for every pair (the "unrelated" default).
    matrix = np.full((10, 10), 0.1)
    # Same genre is perfectly similar.
    np.fill_diagonal(matrix, 1.0)

    # Override specific pairs that have a defined relationship.
    # Each tuple is (genre_index_A, genre_index_B, similarity_score).
    # The matrix is symmetric so we set both [i][j] and [j][i].
    explicit_pairs = [
        (0, 9, 0.6),   # Romance <-> General Fiction
        (0, 4, 0.5),   # Romance <-> Young Adult
        (0, 8, 0.2),   # Romance <-> Self Help
        (1, 6, 0.4),   # Mystery/Thriller <-> History
        (1, 9, 0.4),   # Mystery/Thriller <-> General Fiction
        (1, 2, 0.3),   # Mystery/Thriller <-> SciFi/Fantasy
        (1, 5, 0.2),   # Mystery/Thriller <-> Biography
        (2, 4, 0.6),   # SciFi/Fantasy <-> Young Adult
        (2, 3, 0.4),   # SciFi/Fantasy <-> Children
        (2, 7, 0.4),   # SciFi/Fantasy <-> SciTech
        (3, 4, 0.5),   # Children <-> Young Adult
        (4, 9, 0.5),   # Young Adult <-> General Fiction
        (5, 6, 0.7),   # Biography <-> History
        (5, 7, 0.4),   # Biography <-> SciTech
        (5, 8, 0.3),   # Biography <-> Self Help
        (6, 7, 0.4),   # History <-> SciTech
        (6, 9, 0.3),   # History <-> General Fiction
        (7, 8, 0.3),   # SciTech <-> Self Help
        (8, 9, 0.2),   # Self Help <-> General Fiction
    ]

    for i, j, sim in explicit_pairs:
        matrix[i][j] = sim
        matrix[j][i] = sim

    return matrix


GENRE_SIMILARITY = _build_similarity_matrix()


def genre_to_index(label):
    """Translate Katy's genre label into its matrix index (0-9).

    The spelling must match GENRE_LABELS exactly - that is the agreed
    interface between her classifier and this step. An unknown label
    falls back to General & Contemporary Fiction (index 9) with a loud
    warning, so one upstream typo cannot crash the whole ranking.
    """
    if label in GENRE_TO_INDEX:
        return GENRE_TO_INDEX[label]
    print(f"[WARNING] Unknown genre label {label!r} - "
          f"falling back to 'General & Contemporary Fiction'")
    return GENRE_TO_INDEX["General & Contemporary Fiction"]


# Quick sanity checks on the matrix
assert GENRE_SIMILARITY.shape == (10, 10)
assert np.allclose(GENRE_SIMILARITY, GENRE_SIMILARITY.T), "matrix must be symmetric"
assert np.allclose(np.diag(GENRE_SIMILARITY), 1.0), "diagonal must be 1.0"
print("Genre similarity matrix OK - example: Biography <-> History =",
      GENRE_SIMILARITY[genre_to_index("Biography & Memoir"),
                       genre_to_index("History & Politics")])

Genre similarity matrix OK - example: Biography <-> History = 0.7


In [19]:
from sklearn.preprocessing import StandardScaler

# The exact vector layout agreed with Alexa. Genre travels as its matrix
# index (her ranker looks pairs up in GENRE_SIMILARITY); the two numeric
# features are z-scores from a scaler fitted once on the profile.
VECTOR_COLUMNS = ["genre_index", "known_author", "pageCount_z", "year_z"]


def fit_profile_scaler(profile_df):
    """Fit the StandardScaler once, on the user's profile books only.

    z = (x - mean) / standard deviation, per column, for pageCount and
    year. Fitting on the profile and reusing that same scaler for every
    shelf book keeps all vectors on one common scale; refitting per
    shelf upload would move the goalposts between photos.
    """
    return StandardScaler().fit(profile_df[["pageCount", "year"]])


def is_known_author(book_authors, profile_authors):
    """Return 1.0 if any profile author appears in the book's author
    string, else 0.0 (case-insensitive full-name match)."""
    haystack = str(book_authors).lower()
    return float(any(a.lower() in haystack for a in profile_authors))


def build_feature_vectors(df, genre_labels, profile_authors, scaler):
    """Turn each book row into the four-number vector for Alexa's ranker.

    Parameters
    ----------
    df : pandas.DataFrame
        Clean metadata table (after the missing-data policy, so pageCount
        and year are guaranteed present).
    genre_labels : iterable of str
        One genre label per row, from Katy's classifier, in the agreed
        spelling.
    profile_authors : list of str
        Authors of the books in the user's saved profile.
    scaler : fitted StandardScaler
        From fit_profile_scaler - fitted on the profile, never on the shelf.

    Returns
    -------
    pandas.DataFrame
        One row per book, columns in VECTOR_COLUMNS order, indexed like
        df, ready for .to_numpy().
    """
    z_scores = scaler.transform(df[["pageCount", "year"]])
    vectors = pd.DataFrame({
        "genre_index": [genre_to_index(g) for g in genre_labels],
        "known_author": [is_known_author(a, profile_authors) for a in df["authors"]],
        "pageCount_z": z_scores[:, 0],
        "year_z": z_scores[:, 1],
    }, index=df.index)
    return vectors[VECTOR_COLUMNS]

In [20]:
# --- Demo run on the twelve test books -------------------------------
# In the app, the genre label comes from Katy's classifier. Until that is
# wired in, hand-assigned labels (exact agreed spellings!) stand in so the
# vector step can be tested end to end.
DEMO_GENRES = {
    "Red Rising": "Science Fiction & Fantasy",
    "The Secret History": "Mystery, Thriller & Crime",
    "Klara and the Sun": "Science Fiction & Fantasy",
    "Piranesi": "Science Fiction & Fantasy",
    "Normal People": "General & Contemporary Fiction",
    "Rebecca": "Mystery, Thriller & Crime",
    "Sapiens": "History & Politics",
    "The Alchemist": "General & Contemporary Fiction",
    "Flowers for Algernon": "Science Fiction & Fantasy",
    "The Penelopiad": "General & Contemporary Fiction",
    "Beloved": "General & Contemporary Fiction",
    "A Fine Balance": "General & Contemporary Fiction",
}

# Demo taste profile: the books this user already enjoyed. In the app this
# comes from onboarding (ten books). The profile does two jobs: its
# authors feed the known_author flag, and its numerics fit the scaler.
PROFILE_TITLES = ["Piranesi", "Rebecca", "Klara and the Sun", "The Secret History"]
profile_df = books_clean_df[books_clean_df["query_title"].isin(PROFILE_TITLES)]
profile_authors = list(profile_df["authors"])

scaler = fit_profile_scaler(profile_df)
genre_labels = books_clean_df["query_title"].map(DEMO_GENRES)
vectors = build_feature_vectors(books_clean_df, genre_labels, profile_authors, scaler)

print("Vector order:", ", ".join(VECTOR_COLUMNS))
print("\nThree example vectors:\n")
for example_title in ["Red Rising", "Rebecca", "A Fine Balance"]:
    idx = books_clean_df.index[books_clean_df["query_title"] == example_title][0]
    print(f"{example_title:16s} -> {vectors.loc[idx].to_numpy().round(3)}")

# Hand these to Alexa for a round-trip test 
vectors_out = pd.concat([books_clean_df[["query_title", "authors"]], vectors], axis=1)
vectors_out.to_csv("../data/feature_vectors_demo.csv", index=False)
print("\nSaved to ../data/feature_vectors_demo.csv")
vectors_out

Vector order: genre_index, known_author, pageCount_z, year_z

Three example vectors:

Red Rising       -> [2.    0.    0.287 0.071]
Rebecca          -> [ 1.     1.     0.139 -1.069]
A Fine Balance   -> [ 9.     0.     3.587 -0.499]

Saved to ../data/feature_vectors_demo.csv


,query_title,authors,genre_index,known_author,pageCount_z,year_z
0,Red Rising,Pierce Brown,2,0.0,0.287344,0.071247
1,The Secret History,Donna Tartt,1,1.0,1.544898,-0.926212
2,Klara and the Sun,Kazuo Ishiguro,2,1.0,-0.544264,1.068706
3,Piranesi,Susanna Clarke,2,1.0,-1.139235,0.926212
4,Normal People,Sally Rooney,9,0.0,-0.672724,0.783718
5,Rebecca,Daphne Du Maurier,1,1.0,0.138601,-1.068706
6,Sapiens,Yuval Noah Harari,6,0.0,-0.307627,0.071247
7,The Alchemist,Paulo Coelho,9,0.0,-1.504332,-1.638682
8,Flowers for Algernon,Daniel Keyes,2,0.0,-0.848511,-6.768470
9,The Penelopiad,Margaret Atwood,9,0.0,-1.903233,0.071247
